# External Validation on NHANES 2021-2023

In [1]:
import pandas as pd
import numpy as np
import os, sys, json, joblib
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, matthews_corrcoef, brier_score_loss
from sklearn.metrics import roc_curve

sys.path.append("..")

RAW_PATH = "../data/raw/NHANES_2021_2023"
PROCESSED_DIR = "../data/processed"
MODEL_DIR = "../models"
os.makedirs(RAW_PATH, exist_ok=True)

In [2]:
# Load training master (contains selected features but not latest derived ones)
train_master = pd.read_csv(os.path.join(PROCESSED_DIR, "labeled_master_combined.csv"))

# Define curated feature list (same as before)
SELECTED_FEATURES = [
    "RIDAGEYR", "RIAGENDR", "RIDRETH3", "INDFMPIR", "DMDEDUC2", "DMDMARTL",
    "DMDBORN4", "DMDYRSUS", "INDHHIN2", "DMDHHSIZ", "RIDEXMON",
    "BMXWT", "BMXHT", "BMXBMI", "BMXWAIST", "BMXHIP", "BMXARMC", "BMXARML", "BMXLEG",
    "LBXGH", "LBXGLU",
    "BPXSY1", "BPXDI1", "BPXSY2", "BPXDI2", "BPXSY3", "BPXDI3", "BPXSY4", "BPXDI4",
    "BPXPLS", "BPXPULS",
    "LBXSCR", "LBXSBU", "LBXSAL", "LBXSTP", "LBXSCA", "LBXSNASI", "LBXSKSI",
    "LBXSCLSI", "LBXSTB", "LBXSAPSI", "LBXSATSI", "LBXSASSI", "LBXSGTSI",
    "LBXSPH", "LBXSC3SI", "LBXSUA", "LBXSIR", "LBXSGB", "LBXSCK", "LBXSLDSI",
    "LBXSOSSI",
    "DIQ160", "DID040", "DIQ080",
    "KIQ022", "KIQ025", "KIQ026",
    "SMQ020", "SMD030", "SMQ040", "SMD650", "SMD057", "SMQ890", "SMQ900", "SMQ910",
    "ALQ111", "ALQ121", "ALQ130", "ALQ142", "ALQ151",
    "PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665", "PAD680",
    "DBQ700", "DBQ197", "DBD895", "DBD900", "DBD905", "DBD910",
    "MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F",
    "MCQ160L", "MCQ300C", "MCQ300A",
    "LBXWBCSI", "LBXHGB", "LBXRDW", "LBXPLTSI", "LBXLYPCT", "LBXNEPCT",
    "LBDHDD", "LBXTC", "LBXTR",
    "LBXIN", "LBXHSCRP", "URDACT",
    "HSD010"
]

# ====================================================
# Compute derived features on train_master (same as training)
# ====================================================

# eGFR
def egfr_ckdepi_2021(scr, age, sex):
    scr = np.asarray(scr, dtype=float)
    age = np.asarray(age, dtype=float)
    sex = np.asarray(sex, dtype=float)
    kappa = np.where(sex == 1, 0.9, 0.7)
    alpha = np.where(sex == 1, -0.302, -0.241)
    ratio = scr / kappa
    ratio_min = np.minimum(ratio, 1)
    ratio_max = np.maximum(ratio, 1)
    egfr = 142 * (ratio_min ** alpha) * (ratio_max ** -1.200) * (0.9938 ** age) * np.where(sex == 1, 1.0, 1.012)
    return egfr

train_master["eGFR"] = egfr_ckdepi_2021(train_master["LBXSCR"], train_master["RIDAGEYR"], train_master["RIAGENDR"])

# Mean BP
bp_sys_cols = ["BPXSY1","BPXSY2","BPXSY3","BPXSY4"]
bp_dia_cols = ["BPXDI1","BPXDI2","BPXDI3","BPXDI4"]
train_master["mean_sbp"] = train_master[bp_sys_cols].mean(axis=1, skipna=True)
train_master["mean_dbp"] = train_master[bp_dia_cols].mean(axis=1, skipna=True)

# ASCVD risk (simplified)
def ascvd_risk(row):
    age = row["RIDAGEYR"]
    sex = row["RIAGENDR"]
    tc = row.get("LBXTC", np.nan)
    hdl = row.get("LBDHDD", np.nan)
    sbp = row.get("mean_sbp", np.nan)
    smoker = 1 if row.get("SMQ040", 0) in [1,2] else 0
    diabetic = 1 if row.get("DIQ010", 2) == 1 else 0
    ht_treated = 1 if row.get("BPQ020", 2) == 1 else 0
    risk = age*0.05 + tc*0.002 - hdl*0.003 + sbp*0.004 + smoker*0.15 + diabetic*0.2 + ht_treated*0.1
    return max(0.0, min(risk, 1.0))

train_master["ASCVD_risk"] = train_master.apply(ascvd_risk, axis=1)

# Metabolic syndrome count
def met_syndrome_count(row):
    count = 0
    if (row["RIAGENDR"]==1 and row["BMXWAIST"]>102) or (row["RIAGENDR"]==2 and row["BMXWAIST"]>88): count += 1
    if not pd.isna(row.get("LBXTR", np.nan)) and row["LBXTR"]>=150: count += 1
    if not pd.isna(row.get("LBDHDD", np.nan)):
        hdl_low = (row["LBDHDD"]<40) if row["RIAGENDR"]==1 else (row["LBDHDD"]<50)
        if hdl_low: count += 1
    if not pd.isna(row.get("mean_sbp", np.nan)) and not pd.isna(row.get("mean_dbp", np.nan)):
        if row["mean_sbp"]>=130 or row["mean_dbp"]>=85: count += 1
    if not pd.isna(row.get("LBXGLU", np.nan)) and row["LBXGLU"]>=100: count += 1
    return count

train_master["met_syndrome_count"] = train_master.apply(met_syndrome_count, axis=1)

# eGFR stage
def egfr_stage(egfr):
    if pd.isna(egfr): return np.nan
    if egfr>=90: return 1
    elif egfr>=60: return 2
    elif egfr>=45: return 3
    elif egfr>=30: return 4
    else: return 5

train_master["eGFR_stage"] = train_master["eGFR"].apply(egfr_stage)

# Obesity class
def obesity_class(bmi):
    if pd.isna(bmi): return np.nan
    if bmi<18.5: return 0
    elif bmi<25: return 1
    elif bmi<30: return 2
    elif bmi<35: return 3
    elif bmi<40: return 4
    else: return 5

train_master["obesity_class"] = train_master["BMXBMI"].apply(obesity_class)

# Pack-years
def pack_years(row):
    if pd.isna(row.get("SMQ020", np.nan)) or row["SMQ020"]!=1:
        return 0
    start_age = row.get("SMD030", np.nan)
    current_age = row["RIDAGEYR"]
    if pd.isna(start_age) or start_age<=0:
        return 0
    years_smoked = max(0, current_age - start_age)
    cigs_per_day = row.get("SMD650", 0)
    if pd.isna(cigs_per_day): cigs_per_day = 0
    return (cigs_per_day * years_smoked) / 20

train_master["pack_years"] = train_master.apply(pack_years, axis=1)

# ====================================================
# Compute medians for imputation (selected + derived)
# ====================================================
DERIVED_FEATURES = ["ASCVD_risk", "met_syndrome_count", "eGFR_stage", "obesity_class", "pack_years"]
all_features_for_imputation = SELECTED_FEATURES + DERIVED_FEATURES + ["eGFR"]
train_medians = train_master[all_features_for_imputation].median()
print("Training medians computed for", len(train_medians), "features")

C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_master["ASCVD_risk"] = train_master.apply(ascvd_risk, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:83: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_master["met_syndrome_count"] = train_master.apply(met_syndrome_count, axis=1)
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\4253358614.py:94: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `fra

Training medians computed for 110 features


In [3]:
# Required columns for external validation
# We include SELECTED_FEATURES plus temporary sources needed for derived features.
# RIDEXPRG is needed for diabetes label exclusion.
REQUIRED_COLS = ["SEQN", "cycle"] + SELECTED_FEATURES + ["DIQ010", "BPQ020", "RIDEXPRG"]

files = {
    "DEMO": "DEMO_L.xpt",
    "BMX": "BMX_L.xpt",
    "GHB": "GHB_L.xpt",
    "GLU": "GLU_L.xpt",
    "BPX": "BPXO_L.xpt",   # oscillometric BP
    "BIOPRO": "BIOPRO_L.xpt",
    "DIQ": "DIQ_L.xpt",
    "KIQ_U": "KIQ_U_L.xpt",
    "SMQ": "SMQ_L.xpt",
    "ALQ": "ALQ_L.xpt",
    "PAQ": "PAQ_L.xpt",
    "DBQ": "DBQ_L.xpt",
    "MCQ": "MCQ_L.xpt",
    "BPQ": "BPQ_L.xpt",
    "CBC": "CBC_L.xpt",
    "HDL": "HDL_L.xpt",
    "TCHOL": "TCHOL_L.xpt",
    "TRIGLY": "TRIGLY_L.xpt",
    "INS": "INS_L.xpt",
    "HSCRP": "HSCRP_L.xpt",
    "ALB_CR": "ALB_CR_L.xpt",
    "HSQ": "HSQ_L.xpt",
}

dfs = {}
for comp, fname in files.items():
    path = os.path.join(RAW_PATH, fname)
    if not os.path.exists(path):
        print(f"❌ Missing {fname}")
        continue
    df = pd.read_sas(path)
    df["cycle"] = 3

    # Rename BPXO variables to BPX equivalents
    if comp == "BPX":
        rename_map = {
            "BPXOSY1": "BPXSY1", "BPXODI1": "BPXDI1",
            "BPXOSY2": "BPXSY2", "BPXODI2": "BPXDI2",
            "BPXOSY3": "BPXSY3", "BPXODI3": "BPXDI3",
            "BPXOSY4": "BPXSY4", "BPXODI4": "BPXDI4",
            "BPXOPLS": "BPXPLS", "BPXOPULS": "BPXPULS"
        }
        df.rename(columns=rename_map, inplace=True)

    # Rename 2021-2023 triglycerides variable to match training variable
    if comp == "TRIGLY":
        df.rename(columns={"LBXTLG": "LBXTR"}, inplace=True)

    # Keep only columns that exist in this component
    cols_to_keep = [c for c in REQUIRED_COLS if c in df.columns]
    df = df[cols_to_keep]

    dfs[comp] = df
    print(f"Loaded {comp}: {df.shape}, columns={len(df.columns)}")

# Merge on SEQN and cycle
merged = dfs["DEMO"].copy()
for comp in ["BMX","GHB","GLU","BPX","BIOPRO","DIQ","KIQ_U","SMQ","ALQ","PAQ",
             "DBQ","MCQ","BPQ","CBC","HDL","TCHOL","TRIGLY","INS","HSCRP","ALB_CR","HSQ"]:
    if comp in dfs:
        merged = merged.merge(dfs[comp], on=["SEQN", "cycle"], how="inner")
        print(f"Merged {comp}: {merged.shape}")

print(f"\nFinal external dataframe shape: {merged.shape}")

Loaded DEMO: (11933, 11), columns=11
Loaded BMX: (8860, 10), columns=10
Loaded GHB: (7199, 3), columns=3
Loaded GLU: (3996, 3), columns=3
Loaded BPX: (7801, 8), columns=8
Loaded BIOPRO: (7199, 23), columns=23
Loaded DIQ: (11744, 5), columns=5
Loaded KIQ_U: (7809, 4), columns=4
Loaded SMQ: (9015, 5), columns=5
Loaded ALQ: (6337, 7), columns=7
Loaded PAQ: (8153, 3), columns=3
Loaded DBQ: (11933, 2), columns=2
Loaded MCQ: (11744, 8), columns=8
Loaded BPQ: (8501, 3), columns=3
Loaded CBC: (8727, 8), columns=8
Loaded HDL: (8068, 3), columns=3
Loaded TCHOL: (8068, 3), columns=3
Loaded TRIGLY: (3996, 3), columns=3
Loaded INS: (3996, 3), columns=3
Loaded HSCRP: (8727, 3), columns=3
Loaded ALB_CR: (8493, 3), columns=3
Loaded HSQ: (6615, 2), columns=2
Merged BMX: (8860, 19)
Merged GHB: (7199, 20)
Merged GLU: (3996, 21)
Merged BPX: (3996, 27)
Merged BIOPRO: (3996, 48)
Merged DIQ: (3996, 51)
Merged KIQ_U: (3425, 53)
Merged SMQ: (3425, 56)
Merged ALQ: (3425, 61)
Merged PAQ: (3425, 62)
Merged DBQ: (

In [4]:
# ====================================================
# COMPUTE ALL DERIVED FEATURES ON EXTERNAL DATA
# ====================================================

# Helper to safely get column or NaN
def safe_get(row, col):
    return row[col] if col in row and not pd.isna(row[col]) else np.nan

# eGFR
def egfr_ckdepi_2021(scr, age, sex):
    scr = np.asarray(scr, dtype=float)
    age = np.asarray(age, dtype=float)
    sex = np.asarray(sex, dtype=float)
    kappa = np.where(sex == 1, 0.9, 0.7)
    alpha = np.where(sex == 1, -0.302, -0.241)
    ratio = scr / kappa
    ratio_min = np.minimum(ratio, 1)
    ratio_max = np.maximum(ratio, 1)
    return 142 * (ratio_min ** alpha) * (ratio_max ** -1.200) * (0.9938 ** age) * np.where(sex == 1, 1.0, 1.012)

# Compute eGFR if creatinine, age, sex exist
if {"LBXSCR","RIDAGEYR","RIAGENDR"}.issubset(merged.columns):
    merged["eGFR"] = egfr_ckdepi_2021(merged["LBXSCR"], merged["RIDAGEYR"], merged["RIAGENDR"])
else:
    merged["eGFR"] = np.nan

# Mean BP using available columns
bp_sys_cols = [c for c in ["BPXSY1","BPXSY2","BPXSY3","BPXSY4"] if c in merged.columns]
bp_dia_cols = [c for c in ["BPXDI1","BPXDI2","BPXDI3","BPXDI4"] if c in merged.columns]
merged["mean_sbp"] = merged[bp_sys_cols].mean(axis=1, skipna=True) if bp_sys_cols else np.nan
merged["mean_dbp"] = merged[bp_dia_cols].mean(axis=1, skipna=True) if bp_dia_cols else np.nan

# HOMA-IR = (LBXIN * LBXGLU) / 405
if {"LBXIN","LBXGLU"}.issubset(merged.columns):
    merged["HOMA_IR"] = (merged["LBXIN"] * merged["LBXGLU"]) / 405
else:
    merged["HOMA_IR"] = np.nan

# NLR = neutrophils / lymphocytes
if {"LBXNEPCT","LBXLYPCT"}.issubset(merged.columns):
    merged["NLR"] = merged["LBXNEPCT"] / merged["LBXLYPCT"].replace(0, np.nan)
else:
    merged["NLR"] = np.nan

# WHR = waist / hip
if {"BMXWAIST","BMXHIP"}.issubset(merged.columns):
    merged["WHR"] = merged["BMXWAIST"] / merged["BMXHIP"].replace(0, np.nan)
else:
    merged["WHR"] = np.nan

# ASCVD risk (simplified)
def ascvd_risk(row):
    age = row.get("RIDAGEYR", np.nan)
    sex = row.get("RIAGENDR", np.nan)
    tc = row.get("LBXTC", np.nan)
    hdl = row.get("LBDHDD", np.nan)
    sbp = row.get("mean_sbp", np.nan)
    smoker = 1 if row.get("SMQ040", 0) in [1,2] else 0
    diabetic = 1 if row.get("DIQ010", 2) == 1 else 0
    ht_treated = 1 if row.get("BPQ020", 2) == 1 else 0
    risk = age*0.05 + tc*0.002 - hdl*0.003 + sbp*0.004 + smoker*0.15 + diabetic*0.2 + ht_treated*0.1
    return max(0.0, min(risk, 1.0))

merged["ASCVD_risk"] = merged.apply(ascvd_risk, axis=1)

# Metabolic syndrome component count
def met_syndrome_count(row):
    count = 0
    if (row.get("RIAGENDR", np.nan)==1 and row.get("BMXWAIST", np.nan)>102) or (row.get("RIAGENDR", np.nan)==2 and row.get("BMXWAIST", np.nan)>88):
        count += 1
    if not pd.isna(row.get("LBXTR", np.nan)) and row.get("LBXTR", np.nan)>=150:
        count += 1
    if not pd.isna(row.get("LBDHDD", np.nan)):
        hdl_low = (row.get("LBDHDD", np.nan)<40) if row.get("RIAGENDR", np.nan)==1 else (row.get("LBDHDD", np.nan)<50)
        if hdl_low: count += 1
    if not pd.isna(row.get("mean_sbp", np.nan)) and not pd.isna(row.get("mean_dbp", np.nan)):
        if row.get("mean_sbp", np.nan)>=130 or row.get("mean_dbp", np.nan)>=85:
            count += 1
    if not pd.isna(row.get("LBXGLU", np.nan)) and row.get("LBXGLU", np.nan)>=100:
        count += 1
    return count

merged["met_syndrome_count"] = merged.apply(met_syndrome_count, axis=1)

# eGFR stage
def egfr_stage(egfr):
    if pd.isna(egfr): return np.nan
    if egfr>=90: return 1
    elif egfr>=60: return 2
    elif egfr>=45: return 3
    elif egfr>=30: return 4
    else: return 5

merged["eGFR_stage"] = merged["eGFR"].apply(egfr_stage)

# Obesity class
def obesity_class(bmi):
    if pd.isna(bmi): return np.nan
    if bmi<18.5: return 0
    elif bmi<25: return 1
    elif bmi<30: return 2
    elif bmi<35: return 3
    elif bmi<40: return 4
    else: return 5

merged["obesity_class"] = merged["BMXBMI"].apply(obesity_class)

# Pack-years
def pack_years(row):
    if pd.isna(row.get("SMQ020", np.nan)) or row.get("SMQ020", np.nan)!=1:
        return 0
    start_age = row.get("SMD030", np.nan)
    current_age = row.get("RIDAGEYR", np.nan)
    if pd.isna(start_age) or start_age<=0:
        return 0
    years_smoked = max(0, current_age - start_age)
    cigs_per_day = row.get("SMD650", 0)
    if pd.isna(cigs_per_day): cigs_per_day = 0
    return (cigs_per_day * years_smoked) / 20

merged["pack_years"] = merged.apply(pack_years, axis=1)

print("All derived features computed on external data.")

All derived features computed on external data.


In [5]:
DISEASE_MAP = {
    "diabetes": "label_diabetes",
    "hypertension": "label_hypertension",
    "ckd": "label_ckd",
    "dyslipidemia": "label_dyslipidemia",
    "cvd": "label_cvd",
    "metabolic_syndrome": "label_metabolic_syndrome",
    "liver_disease": "label_liver_disease",
}

with open(os.path.join(PROCESSED_DIR, "feature_names.json")) as f:
    feature_names_dict = json.load(f)

external_predictions = pd.DataFrame({"SEQN": merged["SEQN"], "cycle": merged["cycle"]})

for disease, label_col in DISEASE_MAP.items():
    print(f"Evaluating {disease}...")
    model_path = os.path.join(MODEL_DIR, f"{disease}_final_model.pkl")
    scaler_path = os.path.join(PROCESSED_DIR, f"{disease}_scaler.pkl")
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        print(f"Model/scaler missing for {disease}")
        continue
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)

    feats = feature_names_dict[label_col]

    # Build X_external with all required columns, missing as NaN
    X_external = pd.DataFrame(index=merged.index)
    for col in feats:
        if col in merged.columns:
            X_external[col] = merged[col]
        else:
            X_external[col] = np.nan

    # Impute missing using training medians (computed in Cell 3)
    for col in feats:
        if col in train_medians:
            X_external[col] = X_external[col].fillna(train_medians[col])
        else:
            X_external[col] = X_external[col].fillna(X_external[col].median())

    # Scale
    X_scaled = scaler.transform(X_external)

    # Predict
    y_prob = model.predict_proba(X_scaled)[:, 1]
    external_predictions[f"{disease}_risk"] = y_prob
    print(f"Saved risk scores for {disease}")

# Save predictions
out_path = os.path.join(PROCESSED_DIR, "external_predictions_2021_2023.csv")
external_predictions.to_csv(out_path, index=False)
print(f"External predictions saved to {out_path}")

Evaluating diabetes...
Saved risk scores for diabetes
Evaluating hypertension...


C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py

Saved risk scores for hypertension
Evaluating ckd...
Saved risk scores for ckd
Evaluating dyslipidemia...
Saved risk scores for dyslipidemia
Evaluating cvd...
Saved risk scores for cvd
Evaluating metabolic_syndrome...


C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:34: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = np.nan
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: Performance

Saved risk scores for metabolic_syndrome
Evaluating liver_disease...
Saved risk scores for liver_disease
External predictions saved to ../data/processed\external_predictions_2021_2023.csv


c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
C:\Users\MiR\AppData\Local\Temp\ipykernel_6420\3216404862.py:32: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_external[col] = merged[col]
c:\Mir\OneHealth-ML\.venv\Lib\site-packages\sklearn\utils\validation.py:2820: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(


In [6]:
# ====================================================
# Generate labels for external data using same rules
# ====================================================

# Helper functions (same as training)
def label_diabetes(row):
    if row.get("RIDEXPRG") == 1:
        return np.nan
    if pd.isna(row.get("LBXGH", np.nan)) and pd.isna(row.get("LBXGLU", np.nan)) and pd.isna(row.get("DIQ010", np.nan)):
        return np.nan
    if (row.get("LBXGH", np.nan) >= 6.5) or (row.get("LBXGLU", np.nan) >= 126) or (row.get("DIQ010", np.nan) == 1):
        return 1
    return 0

def label_hypertension(row):
    if pd.isna(row.get("mean_sbp", np.nan)) and pd.isna(row.get("mean_dbp", np.nan)) and pd.isna(row.get("BPQ020", np.nan)):
        return np.nan
    if (row.get("mean_sbp", np.nan) >= 130) or (row.get("mean_dbp", np.nan) >= 80) or (row.get("BPQ020", np.nan) == 1):
        return 1
    return 0

def label_ckd(row):
    if pd.isna(row.get("eGFR", np.nan)) and pd.isna(row.get("KIQ025", np.nan)) and pd.isna(row.get("KIQ022", np.nan)):
        return np.nan
    if (row.get("eGFR", np.nan) < 60) or (row.get("KIQ025", np.nan) == 1) or (row.get("KIQ022", np.nan) == 1):
        return 1
    return 0

def label_dyslipidemia(row):
    if pd.isna(row.get("LBXTC", np.nan)) and pd.isna(row.get("LBDHDD", np.nan)) and pd.isna(row.get("LBXTR", np.nan)):
        return np.nan
    hdl_low = (row.get("LBDHDD", np.nan) < 40) if row.get("RIAGENDR", np.nan) == 1 else (row.get("LBDHDD", np.nan) < 50)
    if (row.get("LBXTC", np.nan) >= 200) or hdl_low or (row.get("LBXTR", np.nan) >= 150):
        return 1
    return 0

def label_cvd(row):
    cols = ["MCQ160B", "MCQ160C", "MCQ160D", "MCQ160E", "MCQ160F"]
    values = [row.get(c, np.nan) for c in cols]
    if all(pd.isna(v) for v in values):
        return np.nan
    if any(v == 1 for v in values):
        return 1
    return 0

def label_metabolic_syndrome(row):
    count = 0
    if (row.get("RIAGENDR", np.nan)==1 and row.get("BMXWAIST", np.nan)>102) or (row.get("RIAGENDR", np.nan)==2 and row.get("BMXWAIST", np.nan)>88):
        count += 1
    if not pd.isna(row.get("LBXTR", np.nan)) and row.get("LBXTR", np.nan)>=150:
        count += 1
    if not pd.isna(row.get("LBDHDD", np.nan)):
        hdl_low = (row.get("LBDHDD", np.nan)<40) if row.get("RIAGENDR", np.nan)==1 else (row.get("LBDHDD", np.nan)<50)
        if hdl_low: count += 1
    if not pd.isna(row.get("mean_sbp", np.nan)) and not pd.isna(row.get("mean_dbp", np.nan)):
        if row.get("mean_sbp", np.nan)>=130 or row.get("mean_dbp", np.nan)>=85:
            count += 1
    if not pd.isna(row.get("LBXGLU", np.nan)) and row.get("LBXGLU", np.nan)>=100:
        count += 1
    if pd.isna(row.get("BMXWAIST")) or pd.isna(row.get("LBXTR")) or pd.isna(row.get("LBDHDD")) or pd.isna(row.get("mean_sbp")) or pd.isna(row.get("LBXGLU")):
        return np.nan
    return 1 if count >= 3 else 0

def label_liver_disease(row):
    if pd.isna(row.get("LBXSATSI", np.nan)) and pd.isna(row.get("LBXSASSI", np.nan)) and pd.isna(row.get("LBXSGTSI", np.nan)) and pd.isna(row.get("BMXBMI", np.nan)):
        return np.nan
    alt_uln = 40 if row.get("RIAGENDR", np.nan)==1 else 30
    ast_uln = 40
    ggt_uln = 50
    enzyme_elevated = (row.get("LBXSATSI", np.nan) > alt_uln) | (row.get("LBXSASSI", np.nan) > ast_uln) | (row.get("LBXSGTSI", np.nan) > ggt_uln)
    bmi_high = row.get("BMXBMI", np.nan) >= 25
    if enzyme_elevated and bmi_high:
        return 1
    return 0

# Apply to external merged dataframe
merged["label_diabetes"] = merged.apply(label_diabetes, axis=1)
merged["label_hypertension"] = merged.apply(label_hypertension, axis=1)
merged["label_ckd"] = merged.apply(label_ckd, axis=1)
merged["label_dyslipidemia"] = merged.apply(label_dyslipidemia, axis=1)
merged["label_cvd"] = merged.apply(label_cvd, axis=1)
merged["label_metabolic_syndrome"] = merged.apply(label_metabolic_syndrome, axis=1)
merged["label_liver_disease"] = merged.apply(label_liver_disease, axis=1)

# ====================================================
# Evaluate external performance
# ====================================================
external_metrics = []

for disease in DISEASE_MAP.keys():
    y_true = merged[f"label_{disease}"].dropna().astype(int)
    # Align predictions with non-NaN labels
    idx = merged[f"label_{disease}"].notna()
    y_prob = external_predictions.loc[idx, f"{disease}_risk"].values
    
    if len(y_true) == 0 or len(np.unique(y_true)) < 2:
        print(f"{disease}: not enough positive cases or all same class")
        continue
    
    auroc = roc_auc_score(y_true, y_prob)
    auprc = average_precision_score(y_true, y_prob)
    # compute F1 at Youden threshold (optional)
    fpr, tpr, thresholds = roc_curve(y_true, y_prob)
    youden_idx = np.argmax(tpr - fpr)
    opt_thresh = thresholds[youden_idx]
    y_pred = (y_prob >= opt_thresh).astype(int)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    external_metrics.append({
        "disease": disease,
        "auroc": auroc,
        "auprc": auprc,
        "f1_youden": f1,
        "mcc_youden": mcc,
        "n_pos": int(y_true.sum()),
        "n_neg": int(len(y_true) - y_true.sum())
    })
    print(f"{disease}: AUROC={auroc:.3f}, AUPRC={auprc:.3f}, F1={f1:.3f}, n_pos={y_true.sum()}, n_neg={len(y_true)-y_true.sum()}")

pd.DataFrame(external_metrics).to_csv(os.path.join(MODEL_DIR, "external_validation_metrics.csv"), index=False)
print("\nExternal validation metrics saved.")

diabetes: AUROC=0.903, AUPRC=0.795, F1=0.681, n_pos=666, n_neg=2731
hypertension: AUROC=0.878, AUPRC=0.913, F1=0.818, n_pos=1935, n_neg=1490
ckd: AUROC=0.873, AUPRC=0.575, F1=0.462, n_pos=307, n_neg=3118
dyslipidemia: AUROC=0.856, AUPRC=0.911, F1=0.767, n_pos=1836, n_neg=1265
cvd: AUROC=0.830, AUPRC=0.424, F1=0.430, n_pos=434, n_neg=2991
metabolic_syndrome: AUROC=0.846, AUPRC=0.703, F1=0.688, n_pos=957, n_neg=1935
liver_disease: AUROC=0.843, AUPRC=0.439, F1=0.421, n_pos=425, n_neg=2990

External validation metrics saved.
